# Deep Past Challenge - Translate Akkadian to English: EDA & Starter Notebook

**Competition:** [Deep Past Challenge - Translate Akkadian to English](https://www.kaggle.com/competitions/deep-past-initiative-machine-translation)  
**Host:** Deep Past Initiative  
**Prize Pool:** $50,000  
**Deadline:** March 23, 2026  
**Type:** Featured Competition  
**Author:** Lorenzo Scaturchio ([lorenzoscaturchio](https://www.kaggle.com/lorenzoscaturchio))

---

## Competition Overview

The Deep Past Challenge is a first-of-its-kind machine learning competition to **translate 4,000-year-old Akkadian cuneiform texts into English**. These are Old Assyrian business records from the ancient city of Kanesh -- contracts, letters, loans, and receipts written in the world's oldest writing system.

### Background
In the early second millennium BCE, merchants from the city of Assur built a vast trade network stretching across the Middle East. They left behind thousands of clay tablets at the site of ancient Kanesh. Most remain untranslated due to the extreme difficulty of cuneiform decipherment.

### Key Facts
- **Input:** Transliterated Akkadian text (cuneiform already converted to Latin characters)
- **Output:** English translation
- **Metric:** Geometric mean of BLEU and chrF++
- **1,319 teams** = Large tier: Bronze top 10% (~132 teams)
- **Niche NLP domain** = specialized knowledge is an advantage

## Medal Analysis & Scoring

With **1,319 teams** this is a large-tier competition:

| Medal | Threshold | Approx. Position |
|-------|-----------|------------------|
| Bronze | Top 10% | Top ~132 |
| Silver | Top 5% | Top ~66 |
| Gold | Top 10 + 0.2% | Top ~13 |

### Evaluation Metric: Geometric Mean of BLEU and chrF++

**Final Score = sqrt(BLEU x chrF++)**

- **BLEU:** Measures word n-gram overlap with reference translation. Sensitive to exact phrasing.
- **chrF++:** Measures character n-gram overlap with word boundary awareness. More forgiving of spelling variations, plurals, and near-misses.
- **Geometric mean:** If either metric is weak, the score collapses. Must balance both.

**Strategy:** Optimize for both word-level accuracy (BLEU) AND character-level similarity (chrF++). Models that produce fluent English with accurate Akkadian terminology will score highest.

---
## Part 1: Environment Setup

In [ ]:
# Install dependencies
!pip install -q transformers datasets sacrebleu sentencepiece tokenizers pandas numpy matplotlib seaborn torch accelerate

In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
sns.set_palette('Set2')

print('Environment ready.')

## Part 2: Understanding the Data

### Data Format
The competition provides transliterated Akkadian text paired with English translations.

**Transliteration** means the cuneiform signs have already been converted to Latin-script representations. Each word is hyphenated to show syllabic structure.

Example:
```
Akkadian: a-na A-shur-i-mi-ti qi2-bi2-ma um-ma Pu-shu-ke-en6-ma
English:  Say to Ashur-imitti, thus says Puzur-Ashur:
```

### Key Linguistic Features
- **Syllabic writing:** Words broken into syllables (e.g., `qi2-bi2-ma` = "say")
- **Determinatives:** Special signs indicating category (divine, city, etc.)
- **Sumerograms:** Sumerian loanwords written in Sumerian
- **Damaged texts:** Many tablets have lacunae (gaps) marked with `[...]`

In [ ]:
# Data paths
DATA_DIR = Path('/kaggle/input/deep-past-initiative-machine-translation')
LOCAL_MODE = not DATA_DIR.exists()

if LOCAL_MODE:
    print("Running in LOCAL MODE with synthetic data")
    print("On Kaggle, real data will be loaded from /kaggle/input/")
    DATA_DIR = Path('synthetic_data')
    DATA_DIR.mkdir(exist_ok=True)
else:
    print(f"Data directory: {DATA_DIR}")
    for f in DATA_DIR.iterdir():
        print(f"  {f.name}")

In [ ]:
# Generate synthetic Akkadian-English parallel corpus for demonstration
# Based on patterns from Old Assyrian commercial texts

if LOCAL_MODE:
    np.random.seed(42)
    
    akkadian_templates = [
        ("a-na {name1} qi2-bi2-ma um-ma {name2}-ma",
         "Say to {ename1}, thus says {ename2}:"),
        ("{n} ma-na {n2} GIN2 KU3.BABBAR s,a-ru-pa-am",
         "{n} minas and {n2} shekels of refined silver"),
        ("{n} GIN2 KU3.BABBAR i-s,e2-er {name1} {name2} i-shu",
         "{name1} owes {n} shekels of silver to {name2}"),
        ("tu-up-pa-am an-ni-a-am i-na ka-ri-im shu-ud-i-ma",
         "Show this tablet at the karum (trade colony)"),
        ("lu-pu-ut-ka li-im-da-ka A-shur",
         "May Ashur keep you in good health"),
        ("{n} TUG2.HI.A shu-qu2-ul-tam u3 ku-ta-nam",
         "{n} textiles, both fine and regular quality"),
        ("an-nu-um tu-up-pu-um sha ra-bi-is, ki-si-im",
         "This is the tablet of the official of the market"),
        ("IGI {name1} DUMU {name2} IGI {name3}",
         "Witnessed by {ename1} son of {ename2}, and {ename3}"),
        ("ni-ish A-shur u3 ni-ish {name1}",
         "By the oath of Ashur and the oath of {ename1}"),
        ("[...] {n} GIN2 KU3.BABBAR [...] ma-la",
         "[...] {n} shekels of silver [...] as much as"),
    ]
    
    akk_names = ['A-shur-i-mi-ti', 'Pu-shu-ke-en6', 'I-di-A-shur', 
                 'E-na-A-shur', 'Dan-A-shur', 'Shu-Ish-tar',
                 'A-shur-na-da', 'Bu-za-zu', 'I-ku-pi2-a',
                 'Shu-A-nim', 'La-qi2-pu-um', 'A-la-hi-im']
    
    eng_names = ['Ashur-imitti', 'Puzur-Ashur', 'Iddi-Ashur',
                 'Ennam-Ashur', 'Dan-Ashur', 'Shu-Ishtar',
                 'Ashur-nada', 'Buzazu', 'Ikuppia',
                 'Shu-Anum', 'Laqipum', 'Alahum']
    
    n_samples = 800
    data = []
    for i in range(n_samples):
        template_idx = np.random.randint(0, len(akkadian_templates))
        akk_template, eng_template = akkadian_templates[template_idx]
        
        name_idx = np.random.choice(len(akk_names), 3, replace=False)
        n_val = np.random.randint(1, 30)
        n2_val = np.random.randint(1, 12)
        
        akk_text = akk_template.format(
            name1=akk_names[name_idx[0]], name2=akk_names[name_idx[1]],
            name3=akk_names[name_idx[2]], n=n_val, n2=n2_val
        )
        eng_text = eng_template.format(
            name1=akk_names[name_idx[0]], name2=akk_names[name_idx[1]],
            name3=akk_names[name_idx[2]], 
            ename1=eng_names[name_idx[0]], ename2=eng_names[name_idx[1]],
            ename3=eng_names[name_idx[2]], n=n_val, n2=n2_val
        )
        
        data.append({
            'id': f'text_{i:04d}',
            'akkadian': akk_text,
            'english': eng_text,
            'template_type': template_idx
        })
    
    df = pd.DataFrame(data)
    train_df = df.iloc[:640].reset_index(drop=True)
    test_df = df.iloc[640:].reset_index(drop=True)
    
    train_df.to_csv(DATA_DIR / 'train.csv', index=False)
    test_df[['id', 'akkadian']].to_csv(DATA_DIR / 'test.csv', index=False)
    test_df[['id']].assign(english='').to_csv(DATA_DIR / 'sample_submission.csv', index=False)
    
    print(f"Generated {len(train_df)} training + {len(test_df)} test samples")
else:
    train_df = pd.read_csv(DATA_DIR / 'train.csv')
    test_df = pd.read_csv(DATA_DIR / 'test.csv')
    print(f"Training samples: {len(train_df)}")
    print(f"Test samples: {len(test_df)}")

In [ ]:
# Display sample data
print("Sample Training Data")
print("=" * 80)
for idx in range(5):
    row = train_df.iloc[idx]
    print(f"\n[{row['id']}]")
    print(f"  Akkadian: {row['akkadian']}")
    print(f"  English:  {row['english']}")

## Part 3: Exploratory Data Analysis

In [ ]:
# Text length analysis
train_df['akk_len'] = train_df['akkadian'].str.len()
train_df['eng_len'] = train_df['english'].str.len()
train_df['akk_words'] = train_df['akkadian'].str.split().str.len()
train_df['eng_words'] = train_df['english'].str.split().str.len()
train_df['length_ratio'] = train_df['eng_len'] / train_df['akk_len'].clip(1)

print("Text Length Statistics")
print("=" * 60)
print(f"\nAkkadian text:")
print(f"  Characters: mean={train_df['akk_len'].mean():.1f}, "
      f"median={train_df['akk_len'].median():.1f}, "
      f"range=[{train_df['akk_len'].min()}, {train_df['akk_len'].max()}]")
print(f"  Words: mean={train_df['akk_words'].mean():.1f}, "
      f"median={train_df['akk_words'].median():.1f}")
print(f"\nEnglish text:")
print(f"  Characters: mean={train_df['eng_len'].mean():.1f}, "
      f"median={train_df['eng_len'].median():.1f}, "
      f"range=[{train_df['eng_len'].min()}, {train_df['eng_len'].max()}]")
print(f"  Words: mean={train_df['eng_words'].mean():.1f}, "
      f"median={train_df['eng_words'].median():.1f}")
print(f"\nLength ratio (Eng/Akk): mean={train_df['length_ratio'].mean():.2f}")

In [ ]:
# Visualize text length distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].hist(train_df['akk_len'], bins=40, color='#e74c3c', alpha=0.7, edgecolor='white')
axes[0, 0].set_title('Akkadian Text Length (chars)', fontweight='bold')
axes[0, 0].set_xlabel('Characters')
axes[0, 0].set_ylabel('Count')
axes[0, 0].axvline(train_df['akk_len'].mean(), color='black', linestyle='--', 
                     label=f'Mean: {train_df["akk_len"].mean():.0f}')
axes[0, 0].legend()

axes[0, 1].hist(train_df['eng_len'], bins=40, color='#3498db', alpha=0.7, edgecolor='white')
axes[0, 1].set_title('English Text Length (chars)', fontweight='bold')
axes[0, 1].set_xlabel('Characters')
axes[0, 1].axvline(train_df['eng_len'].mean(), color='black', linestyle='--',
                     label=f'Mean: {train_df["eng_len"].mean():.0f}')
axes[0, 1].legend()

axes[0, 2].hist(train_df['length_ratio'], bins=40, color='#2ecc71', alpha=0.7, edgecolor='white')
axes[0, 2].set_title('Length Ratio (English/Akkadian)', fontweight='bold')
axes[0, 2].set_xlabel('Ratio')
axes[0, 2].axvline(train_df['length_ratio'].mean(), color='black', linestyle='--',
                     label=f'Mean: {train_df["length_ratio"].mean():.2f}')
axes[0, 2].legend()

axes[1, 0].hist(train_df['akk_words'], bins=30, color='#e74c3c', alpha=0.7, edgecolor='white')
axes[1, 0].set_title('Akkadian Word Count', fontweight='bold')
axes[1, 0].set_xlabel('Words')

axes[1, 1].hist(train_df['eng_words'], bins=30, color='#3498db', alpha=0.7, edgecolor='white')
axes[1, 1].set_title('English Word Count', fontweight='bold')
axes[1, 1].set_xlabel('Words')

axes[1, 2].scatter(train_df['akk_words'], train_df['eng_words'], alpha=0.3, s=20, color='#9b59b6')
axes[1, 2].set_title('Akkadian vs English Word Count', fontweight='bold')
axes[1, 2].set_xlabel('Akkadian Words')
axes[1, 2].set_ylabel('English Words')
z = np.polyfit(train_df['akk_words'], train_df['eng_words'], 1)
p = np.poly1d(z)
x_range = np.linspace(train_df['akk_words'].min(), train_df['akk_words'].max(), 100)
axes[1, 2].plot(x_range, p(x_range), 'r--', linewidth=2, label=f'y={z[0]:.2f}x+{z[1]:.2f}')
axes[1, 2].legend()

plt.suptitle('Text Length Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('length_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Vocabulary analysis - Akkadian side
all_akk_tokens = []
for text in train_df['akkadian']:
    tokens = text.split()
    all_akk_tokens.extend(tokens)

akk_vocab = Counter(all_akk_tokens)
print(f"Akkadian Vocabulary Analysis")
print(f"=" * 50)
print(f"Total tokens: {len(all_akk_tokens):,}")
print(f"Unique tokens: {len(akk_vocab):,}")
print(f"Type-token ratio: {len(akk_vocab)/len(all_akk_tokens):.4f}")
print(f"\nTop 20 most frequent tokens:")
for token, count in akk_vocab.most_common(20):
    print(f"  {token:30s} {count:5d} ({count/len(all_akk_tokens):.2%})")

In [ ]:
# Vocabulary analysis - English side
all_eng_tokens = []
for text in train_df['english']:
    tokens = text.lower().split()
    all_eng_tokens.extend(tokens)

eng_vocab = Counter(all_eng_tokens)
print(f"English Vocabulary Analysis")
print(f"=" * 50)
print(f"Total tokens: {len(all_eng_tokens):,}")
print(f"Unique tokens: {len(eng_vocab):,}")
print(f"Type-token ratio: {len(eng_vocab)/len(all_eng_tokens):.4f}")
print(f"\nTop 20 most frequent tokens:")
for token, count in eng_vocab.most_common(20):
    print(f"  {token:30s} {count:5d} ({count/len(all_eng_tokens):.2%})")

In [ ]:
# Visualize vocabulary distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

akk_counts = sorted(akk_vocab.values(), reverse=True)
eng_counts = sorted(eng_vocab.values(), reverse=True)

axes[0].loglog(range(1, len(akk_counts)+1), akk_counts, 'r-', alpha=0.7, label='Akkadian')
axes[0].loglog(range(1, len(eng_counts)+1), eng_counts, 'b-', alpha=0.7, label='English')
axes[0].set_title("Zipf's Law: Token Frequency Distribution", fontweight='bold')
axes[0].set_xlabel('Rank (log)')
axes[0].set_ylabel('Frequency (log)')
axes[0].legend()

top_akk = akk_vocab.most_common(15)
axes[1].barh([t[0] for t in top_akk], [t[1] for t in top_akk], color='#e74c3c', alpha=0.8)
axes[1].set_title('Top 15 Akkadian Tokens', fontweight='bold')
axes[1].set_xlabel('Frequency')
axes[1].invert_yaxis()

top_eng = eng_vocab.most_common(15)
axes[2].barh([t[0] for t in top_eng], [t[1] for t in top_eng], color='#3498db', alpha=0.8)
axes[2].set_title('Top 15 English Tokens', fontweight='bold')
axes[2].set_xlabel('Frequency')
axes[2].invert_yaxis()

plt.suptitle('Vocabulary Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('vocabulary_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Akkadian-specific feature analysis
features = {
    'has_numbers': train_df['akkadian'].str.contains(r'\d', regex=True).sum(),
    'has_sumerograms': train_df['akkadian'].str.contains(r'[A-Z]{2,}', regex=True).sum(),
    'has_lacunae': train_df['akkadian'].str.contains(r'\[', regex=True).sum(),
    'has_determinatives': train_df['akkadian'].str.contains(r'\b[df]\b', regex=True).sum(),
    'has_hyphens': train_df['akkadian'].str.contains(r'-', regex=True).sum(),
}

print("Akkadian Text Features")
print("=" * 50)
for feature, count in features.items():
    pct = count / len(train_df) * 100
    print(f"  {feature:25s}: {count:5d} ({pct:.1f}%)")

syllable_counts = []
for text in train_df['akkadian']:
    syllables = text.count('-')
    syllable_counts.append(syllables)

print(f"\nSyllabic Hyphens per Text:")
print(f"  Mean: {np.mean(syllable_counts):.1f}")
print(f"  Median: {np.median(syllable_counts):.1f}")
print(f"  Range: [{min(syllable_counts)}, {max(syllable_counts)}]")

In [ ]:
# Visualize Akkadian features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

feat_names = list(features.keys())
feat_pcts = [v / len(train_df) * 100 for v in features.values()]
colors = sns.color_palette('Set2', len(feat_names))

axes[0].barh(feat_names, feat_pcts, color=colors)
axes[0].set_xlabel('Percentage of Texts (%)')
axes[0].set_title('Akkadian Text Feature Prevalence', fontweight='bold')
for i, v in enumerate(feat_pcts):
    axes[0].text(v + 0.5, i, f'{v:.1f}%', va='center')

axes[1].hist(syllable_counts, bins=30, color='#9b59b6', alpha=0.7, edgecolor='white')
axes[1].set_title('Syllabic Hyphens per Text', fontweight='bold')
axes[1].set_xlabel('Number of Hyphens')
axes[1].set_ylabel('Count')
axes[1].axvline(np.mean(syllable_counts), color='red', linestyle='--',
                 label=f'Mean: {np.mean(syllable_counts):.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('akkadian_features.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 4: Understanding the Evaluation Metric

### Geometric Mean of BLEU and chrF++

**Score = sqrt(BLEU * chrF++)**

This forces balance between:
- **BLEU** (word-level precision): Rewards exact word matches and n-gram overlap
- **chrF++** (character-level recall): More forgiving, catches partial matches

If BLEU = 0.9 and chrF++ = 0.1, score = sqrt(0.09) = 0.3 (penalized!)

In [ ]:
import sacrebleu

def compute_competition_score(predictions, references):
    """Compute the competition metric: geometric mean of BLEU and chrF++."""
    bleu = sacrebleu.corpus_bleu(predictions, [references])
    chrf = sacrebleu.corpus_chrf(predictions, [references], word_order=2)
    
    bleu_score = bleu.score / 100
    chrf_score = chrf.score / 100
    combined = np.sqrt(max(bleu_score, 1e-10) * max(chrf_score, 1e-10))
    
    return {
        'bleu': bleu_score,
        'chrf_pp': chrf_score,
        'combined': combined,
        'bleu_detail': str(bleu),
        'chrf_detail': str(chrf),
    }


refs = ["Say to Ashur-imitti, thus says Puzur-Ashur:"]
examples = {
    'Perfect': ["Say to Ashur-imitti, thus says Puzur-Ashur:"],
    'Good': ["Tell Ashur-imitti, thus speaks Puzur-Ashur:"],
    'Partial': ["Say to Ashur-imitti"],
    'Wrong': ["The silver was delivered to the market"],
    'Empty': [""],
}

print("Metric Sensitivity Analysis")
print("=" * 70)
print(f"Reference: {refs[0]}")
print()
for name, pred in examples.items():
    score = compute_competition_score(pred, refs)
    print(f"  {name:12s} -> BLEU={score['bleu']:.4f}, chrF++={score['chrf_pp']:.4f}, "
          f"Combined={score['combined']:.4f}")
    print(f"               Text: {pred[0][:60]}")

In [ ]:
# Visualize the geometric mean penalty
bleu_range = np.linspace(0.01, 1.0, 100)
chrf_range = np.linspace(0.01, 1.0, 100)
B, C = np.meshgrid(bleu_range, chrf_range)
combined = np.sqrt(B * C)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

im = axes[0].contourf(B, C, combined, levels=20, cmap='viridis')
axes[0].set_xlabel('BLEU Score', fontsize=12)
axes[0].set_ylabel('chrF++ Score', fontsize=12)
axes[0].set_title('Combined Score = sqrt(BLEU * chrF++)', fontweight='bold')
plt.colorbar(im, ax=axes[0], label='Combined Score')
axes[0].plot([0, 1], [0, 1], 'r--', alpha=0.5, label='Equal scores')
axes[0].legend()

scenarios = [
    ('Balanced\n(0.5, 0.5)', 0.5, 0.5),
    ('Balanced\n(0.7, 0.7)', 0.7, 0.7),
    ('Imbalanced\n(0.9, 0.3)', 0.9, 0.3),
    ('Imbalanced\n(0.3, 0.9)', 0.3, 0.9),
    ('High BLEU\n(0.8, 0.4)', 0.8, 0.4),
    ('High chrF\n(0.4, 0.8)', 0.4, 0.8),
]

names = [s[0] for s in scenarios]
scores = [np.sqrt(s[1] * s[2]) for s in scenarios]
bleu_vals = [s[1] for s in scenarios]
chrf_vals = [s[2] for s in scenarios]

x = np.arange(len(names))
w = 0.25
axes[1].bar(x - w, bleu_vals, w, label='BLEU', color='#e74c3c', alpha=0.8)
axes[1].bar(x, chrf_vals, w, label='chrF++', color='#3498db', alpha=0.8)
axes[1].bar(x + w, scores, w, label='Combined', color='#2ecc71', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, fontsize=9)
axes[1].set_ylabel('Score')
axes[1].set_title('Geometric Mean Penalizes Imbalance', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('metric_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Key insight: Balanced improvement across BOTH metrics beats high score on just one.")

## Part 5: Baseline Model - Seq2Seq Translation

### Approach Options
1. **Fine-tuned encoder-decoder** (mT5, NLLB, MarianMT) -- Standard MT approach
2. **Character-level model** (ByT5) -- Good for morphologically complex languages
3. **LLM prompting** (Gemma, Llama) -- Zero/few-shot translation with context
4. **Hybrid:** Fine-tuned model + LLM reranking

We start with **ByT5-base** as it operates at the character level, which is ideal for:
- Handling syllabic Akkadian text with hyphens
- Capturing morphological patterns
- Being robust to rare tokens and transliteration variants

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class AkkadianDataset(Dataset):
    """Dataset for Akkadian-English translation."""
    
    def __init__(self, df, tokenizer, max_source_len=256, max_target_len=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_source_len = max_source_len
        self.max_target_len = max_target_len
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        source = f"translate Akkadian to English: {row['akkadian']}"
        
        source_encoding = self.tokenizer(
            source, max_length=self.max_source_len, 
            padding='max_length', truncation=True, return_tensors='pt'
        )
        
        result = {
            'input_ids': source_encoding['input_ids'].squeeze(),
            'attention_mask': source_encoding['attention_mask'].squeeze(),
        }
        
        if 'english' in row:
            target = row['english']
            target_encoding = self.tokenizer(
                target, max_length=self.max_target_len,
                padding='max_length', truncation=True, return_tensors='pt'
            )
            labels = target_encoding['input_ids'].squeeze()
            labels[labels == self.tokenizer.pad_token_id] = -100
            result['labels'] = labels
        
        return result


print("Dataset class defined.")
print("Input format: 'translate Akkadian to English: <akkadian_text>'")

In [ ]:
# Load ByT5 model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = 'google/byt5-small'  # Start small for demo; use byt5-base for competition

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model: {MODEL_NAME}")
    print(f"Parameters: {total_params:,}")
    print(f"Model size: ~{total_params * 4 / 1e9:.2f} GB (float32)")
    
    sample = "translate Akkadian to English: a-na A-shur-i-mi-ti qi2-bi2-ma"
    tokens = tokenizer(sample, return_tensors='pt')
    print(f"\nSample tokenization:")
    print(f"  Input: {sample}")
    print(f"  Token count: {tokens['input_ids'].shape[1]}")
except Exception as e:
    print(f"Model loading note: {e}")
    print("On Kaggle, the model will load from Kaggle Models.")
    tokenizer = None
    model = None

In [ ]:
# Training configuration
training_config = {
    'model': MODEL_NAME,
    'max_source_length': 256,
    'max_target_length': 256,
    'batch_size': 8,
    'learning_rate': 3e-4,
    'epochs': 10,
    'warmup_steps': 100,
    'weight_decay': 0.01,
    'gradient_accumulation': 4,
    'fp16': True,
    'beam_size': 5,
    'length_penalty': 1.0,
}

print("Training Configuration")
print("=" * 50)
for key, value in training_config.items():
    print(f"  {key}: {value}")

In [ ]:
# Training loop
def train_translation_model(model, tokenizer, train_df, epochs=3, lr=3e-4, batch_size=8, device='cpu'):
    """Fine-tune a seq2seq model on Akkadian-English pairs."""
    if model is None:
        print("Model not loaded. Skipping training (will work on Kaggle).")
        return None, {'train_loss': [0.5, 0.3, 0.2]}
    
    model = model.to(device)
    train_dataset = AkkadianDataset(train_df, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    history = {'train_loss': []}
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        n_batches = 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            
            total_loss += loss.item()
            n_batches += 1
        
        scheduler.step()
        avg_loss = total_loss / max(n_batches, 1)
        history['train_loss'].append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")
    
    return model, history


val_size = int(len(train_df) * 0.1)
val_df = train_df.iloc[-val_size:].reset_index(drop=True)
train_split = train_df.iloc[:-val_size].reset_index(drop=True)

print(f"Training split: {len(train_split)} samples")
print(f"Validation split: {len(val_df)} samples")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

model, history = train_translation_model(
    model, tokenizer, train_split, epochs=3, lr=3e-4, batch_size=4, device=device
)

In [ ]:
# Plot training history
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(history['train_loss'])+1), history['train_loss'], 'b-o', 
        markersize=8, linewidth=2)
ax.set_title('Training Loss', fontweight='bold', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 6: Inference & Evaluation

In [ ]:
def translate_batch(model, tokenizer, texts, device='cpu', 
                    max_length=256, num_beams=5, length_penalty=1.0):
    """Translate a batch of Akkadian texts to English."""
    if model is None:
        return [f"[Placeholder for: {t[:50]}...]" for t in texts]
    
    model.eval()
    prefixed = [f"translate Akkadian to English: {t}" for t in texts]
    
    inputs = tokenizer(
        prefixed, max_length=max_length, padding=True, 
        truncation=True, return_tensors='pt'
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_length=max_length, num_beams=num_beams,
            length_penalty=length_penalty, early_stopping=True,
        )
    
    translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return translations


print("Running validation inference...")
val_predictions = []
val_references = []

for i in range(0, len(val_df), 8):
    batch = val_df.iloc[i:i+8]
    preds = translate_batch(model, tokenizer, batch['akkadian'].tolist(), device=device)
    val_predictions.extend(preds)
    val_references.extend(batch['english'].tolist())

print("\nSample Translations")
print("=" * 80)
for i in range(min(5, len(val_predictions))):
    print(f"\n[{i+1}]")
    print(f"  Akkadian:   {val_df.iloc[i]['akkadian'][:80]}")
    print(f"  Reference:  {val_references[i][:80]}")
    print(f"  Prediction: {val_predictions[i][:80]}")

In [ ]:
# Compute validation metrics
try:
    val_score = compute_competition_score(val_predictions, val_references)
    print("Validation Metrics")
    print("=" * 50)
    print(f"  BLEU:     {val_score['bleu']:.4f}")
    print(f"  chrF++:   {val_score['chrf_pp']:.4f}")
    print(f"  Combined: {val_score['combined']:.4f}")
except Exception as e:
    print(f"Metric computation note: {e}")

## Part 7: Advanced Techniques

In [ ]:
# Data augmentation for low-resource MT

def augment_akkadian(text, english, augment_type='swap'):
    """Augment Akkadian-English pairs to increase training data."""
    if augment_type == 'swap':
        words = text.split()
        if len(words) > 2:
            i = np.random.randint(0, len(words) - 1)
            words[i], words[i+1] = words[i+1], words[i]
        return ' '.join(words), english
    
    elif augment_type == 'noise':
        noise_map = {'sh': 'sz', 'sz': 'sh', 'ts': 'z', 'z': 'ts', '2': '', '3': ''}
        result = text
        for old, new in noise_map.items():
            if np.random.random() > 0.7 and old in result:
                result = result.replace(old, new, 1)
        return result, english
    
    elif augment_type == 'dropout':
        words = text.split()
        if len(words) > 3:
            drop_idx = np.random.randint(0, len(words))
            words[drop_idx] = '[...]'
        return ' '.join(words), english
    
    return text, english


sample_akk = train_df.iloc[0]['akkadian']
sample_eng = train_df.iloc[0]['english']

print("Data Augmentation Examples")
print("=" * 60)
print(f"Original: {sample_akk}")
print()
for aug_type in ['swap', 'noise', 'dropout']:
    aug_akk, aug_eng = augment_akkadian(sample_akk, sample_eng, aug_type)
    print(f"  {aug_type:10s}: {aug_akk}")

In [ ]:
# LLM-based translation approach (alternative baseline)

def create_few_shot_prompt(test_text, examples, n_shots=3):
    """Create a few-shot prompt for LLM-based Akkadian translation."""
    prompt = """You are an expert Assyriologist translating Old Assyrian texts from the ancient city of Kanesh.
These are transliterated cuneiform texts from approximately 1950-1740 BCE.

Key conventions:
- Words are hyphenated by syllable (e.g., qi2-bi2-ma = "say")
- UPPERCASE words are Sumerograms (Sumerian loanwords)
- KU3.BABBAR = silver, GIN2 = shekel, TUG2 = textile
- [...] indicates damaged/missing text on the tablet
- Numbers after letters (e.g., bi2) disambiguate similar cuneiform signs

Translate the following Akkadian texts to English:

"""
    for i, (akk, eng) in enumerate(examples[:n_shots]):
        prompt += f"Example {i+1}:\nAkkadian: {akk}\nEnglish: {eng}\n\n"
    
    prompt += f"Now translate:\nAkkadian: {test_text}\nEnglish:"
    return prompt


examples = list(zip(train_df['akkadian'][:3], train_df['english'][:3]))
test_text = val_df.iloc[0]['akkadian']
prompt = create_few_shot_prompt(test_text, examples)

print("Few-Shot Translation Prompt")
print("=" * 60)
print(prompt[:1200])
print("...")

In [ ]:
# MBR (Minimum Bayes Risk) ensemble strategy

def ensemble_translations(candidates, references=None):
    """
    Select best translation using MBR decoding.
    Picks the candidate most similar to all others (consensus).
    """
    if len(candidates) == 1:
        return candidates[0]
    
    best_score = -1
    best_idx = 0
    
    for i, cand in enumerate(candidates):
        others = [c for j, c in enumerate(candidates) if j != i]
        avg_score = 0
        for other in others:
            try:
                score = sacrebleu.corpus_chrf([cand], [[other]]).score
                avg_score += score
            except Exception:
                avg_score += 0
        avg_score /= max(len(others), 1)
        
        if avg_score > best_score:
            best_score = avg_score
            best_idx = i
    
    return candidates[best_idx]


demo_candidates = [
    "Say to Ashur-imitti, thus says Puzur-Ashur:",
    "Tell Ashur-imitti: thus speaks Puzur-Ashur -",
    "To Ashur-imitti say, thus is Puzur-Ashur speaking:",
]

best = ensemble_translations(demo_candidates)
print("MBR Ensemble Selection")
print("=" * 50)
for i, cand in enumerate(demo_candidates):
    marker = " <-- SELECTED" if cand == best else ""
    print(f"  Candidate {i+1}: {cand}{marker}")

## Part 8: Submission Generation

In [ ]:
def generate_submission(model, tokenizer, test_df, output_path='submission.csv', device='cpu'):
    """Generate competition submission file."""
    all_predictions = []
    batch_size = 16
    
    for i in range(0, len(test_df), batch_size):
        batch = test_df.iloc[i:i+batch_size]
        preds = translate_batch(
            model, tokenizer, batch['akkadian'].tolist(), 
            device=device, num_beams=5, length_penalty=1.0
        )
        all_predictions.extend(preds)
        if (i // batch_size) % 10 == 0:
            print(f"  Processed {i + len(batch)}/{len(test_df)} texts")
    
    submission = pd.DataFrame({'id': test_df['id'], 'english': all_predictions})
    submission.to_csv(output_path, index=False)
    print(f"\nSubmission saved to {output_path}")
    print(f"Shape: {submission.shape}")
    return submission


if LOCAL_MODE:
    test_input = pd.read_csv(DATA_DIR / 'test.csv')
else:
    test_input = test_df

print(f"Generating submission for {len(test_input)} test texts...")
submission = generate_submission(model, tokenizer, test_input, device=device)
submission.head()

## Part 9: Model Comparison Framework

In [ ]:
# Model comparison
model_configs = {
    'ByT5-small': {'params': '300M', 'type': 'Character-level T5',
                    'est_bleu': 0.15, 'est_chrf': 0.35},
    'ByT5-base': {'params': '580M', 'type': 'Character-level T5',
                   'est_bleu': 0.22, 'est_chrf': 0.42},
    'mT5-base': {'params': '580M', 'type': 'Multilingual T5',
                  'est_bleu': 0.18, 'est_chrf': 0.38},
    'NLLB-600M': {'params': '600M', 'type': 'No Language Left Behind',
                   'est_bleu': 0.20, 'est_chrf': 0.40},
    'Gemma-2B': {'params': '2B', 'type': 'Decoder-only LLM (few-shot)',
                  'est_bleu': 0.12, 'est_chrf': 0.30},
}

print("Model Comparison")
print("=" * 70)
for name, cfg in model_configs.items():
    combined = np.sqrt(cfg['est_bleu'] * cfg['est_chrf'])
    print(f"  {name:15s} ({cfg['params']:>5s}) | {cfg['type']:30s} | "
          f"BLEU={cfg['est_bleu']:.2f} chrF={cfg['est_chrf']:.2f} Combined={combined:.3f}")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

names = list(model_configs.keys())
bleu_scores = [model_configs[n]['est_bleu'] for n in names]
chrf_scores = [model_configs[n]['est_chrf'] for n in names]
combined_scores = [np.sqrt(b * c) for b, c in zip(bleu_scores, chrf_scores)]

x = np.arange(len(names))
w = 0.25
axes[0].bar(x - w, bleu_scores, w, label='BLEU', color='#e74c3c', alpha=0.8)
axes[0].bar(x, chrf_scores, w, label='chrF++', color='#3498db', alpha=0.8)
axes[0].bar(x + w, combined_scores, w, label='Combined', color='#2ecc71', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=45, ha='right')
axes[0].set_ylabel('Score')
axes[0].set_title('Estimated Model Performance', fontweight='bold')
axes[0].legend()

colors = sns.color_palette('Set2', len(names))
for i, name in enumerate(names):
    axes[1].scatter(bleu_scores[i], chrf_scores[i], s=200, c=[colors[i]], 
                     zorder=5, edgecolors='black', linewidth=1)
    axes[1].annotate(name, (bleu_scores[i], chrf_scores[i]), 
                      textcoords="offset points", xytext=(10, 5), fontsize=9)

for score_level in [0.15, 0.20, 0.25, 0.30]:
    b_range = np.linspace(0.05, 0.5, 100)
    c_range = score_level**2 / b_range
    valid = c_range <= 0.6
    axes[1].plot(b_range[valid], c_range[valid], '--', color='gray', alpha=0.4)

axes[1].set_xlabel('BLEU Score')
axes[1].set_ylabel('chrF++ Score')
axes[1].set_title('BLEU vs chrF++ (iso-score curves)', fontweight='bold')

plt.suptitle('Model Comparison for Akkadian Translation', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 10: Akkadian Language Primer

In [ ]:
# Akkadian language reference
akkadian_reference = {
    'Common Sumerograms': {
        'KU3.BABBAR': 'silver', 'KU3.GI': 'gold',
        'GIN2': 'shekel (~8.3g)', 'MA.NA': 'mina (60 shekels)',
        'TUG2': 'textile/cloth', 'GU2.UN': 'talent (60 minas)',
        'DUMU': 'son (of)', 'IGI': 'witness / before', 'ITI': 'month',
    },
    'Common Akkadian Phrases': {
        'qi2-bi2-ma': 'say / speak', 'um-ma ... -ma': 'thus says ...',
        'a-na': 'to / for', 'i-na': 'in / at', 'u3': 'and',
        'la': 'not / no', 'shu-ma': 'if',
        'i-shu': 'he has / owes', 'i-di-in': 'he gave', 'i-pu-ul': 'he paid',
    },
    'Text Types': {
        'Letters': 'Start with "a-na X qi2-bi2-ma um-ma Y-ma"',
        'Debt notes': 'Record amounts owed between parties',
        'Contracts': 'Include witness clauses (IGI)',
        'Receipts': 'Record goods received',
        'Legal docs': 'Include oath formulas (ni-ish)',
    }
}

print("Akkadian Language Reference")
print("=" * 60)
for section, items in akkadian_reference.items():
    print(f"\n{section}:")
    print("-" * 40)
    for key, value in items.items():
        print(f"  {key:25s} = {value}")

In [ ]:
# Build glossary from training data
print("Building Akkadian-English Glossary from Training Data")
print("=" * 60)

word_pairs = Counter()
for _, row in train_df.iterrows():
    akk_words = row['akkadian'].split()
    eng_words = row['english'].lower().split()
    for aw in akk_words:
        for ew in eng_words:
            if len(aw) > 2 and len(ew) > 2:
                word_pairs[(aw, ew)] += 1

print("\nTop 20 Akkadian-English Word Associations:")
for (akk, eng), count in word_pairs.most_common(20):
    print(f"  {akk:25s} <-> {eng:20s}  (count: {count})")

---
## Part 11: Strategic Insights & Medal Path

### Competition Strategy

| Phase | Timeline | Goal | Approach |
|-------|----------|------|----------|
| 1. Baseline | Week 1-2 | ByT5-base fine-tuned | Standard MT training |
| 2. Multi-model | Week 3-4 | Try mT5, NLLB, Gemma | Compare architectures |
| 3. Augmentation | Week 5-6 | Expand training data | Back-translation, noise |
| 4. Ensemble | Week 7-8 | MBR ensemble + glossary | Combine best models |
| 5. Polish | Final week | Optimize beam search, TTA | Maximize both metrics |

### Key Strategies

1. **Character-level models (ByT5)** are ideal for Akkadian's syllabic structure
2. **Domain glossary injection** -- add known Sumerogram translations as constraints
3. **Constrained decoding** -- force output to include known entity translations
4. **Back-translation augmentation** -- generate more training pairs using reverse model
5. **MBR decoding** -- select consensus translation from multiple beam candidates
6. **Balance BLEU and chrF++** -- geometric mean punishes imbalance severely

### Niche Advantage
This is an extremely specialized NLP task. Competitors with understanding of Akkadian grammar, custom preprocessing for Sumerograms, and domain-specific data augmentation will have a significant edge.

In [ ]:
# Final summary
print("\n" + "="*60)
print("Deep Past Challenge - Akkadian Translation - Summary")
print("="*60)
print(f"Competition: Deep Past Challenge - Translate Akkadian to English")
print(f"Prize Pool: $50,000")
print(f"Deadline: March 23, 2026")
print(f"Teams: ~1,319 (Large tier)")
print(f"Medal Thresholds: Bronze ~132nd, Silver ~66th, Gold ~13th")
print(f"\nMetric: sqrt(BLEU * chrF++)")
print(f"Baseline: ByT5-base fine-tuned on Akkadian-English pairs")
print(f"Key Insight: Character-level models + domain glossary + MBR ensemble")
print(f"\nAuthor: Lorenzo Scaturchio")
print(f"Profile: kaggle.com/lorenzoscaturchio")
print("="*60)